To use Microsoft Agent Framework with Azure OpenAI, you need to install the following Python packages:

In [ ]:
%pip install agent-framework --pre

## Create the agent

First, create a chat client for communicating with Azure OpenAI and use the same login as you used when authenticating with the Azure CLI in the Prerequisites step.
Then, create the agent, providing instructions and a name for the agent.

In [4]:
import os

os.environ["NO_PROXY"] = "*"

In [ ]:
%pip install agent-framework==1.0.0b251216
# %pip install agent-framework --pre

In [5]:
from dotenv import load_dotenv
import os

if os.path.exists(".env"):
    load_dotenv(override=True)

In [6]:
# import asyncio
from agent_framework.azure import AzureOpenAIChatClient
# from azure.identity import AzureCliCredential

agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    instructions="You are good at telling jokes.",
    name="Joker"
)
# agent = AzureOpenAIChatClient(deployment_name="gpt-4o-mini", credential=AzureCliCredential()).create_agent(
#     instructions="You are good at telling jokes.",
#     name="Joker"
# )

result = await agent.run("Tell me a joke about a pirate.")
print(result.text)

Why did the pirate go to school?

Because he wanted to improve his "arrrticulation"!


This demo shows you how to use images with an agent, allowing the agent to analyze and respond to image content.

In [7]:
agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    name="VisionAgent",
    instructions="You are a helpful agent that can analyze images"
)

from agent_framework import ChatMessage, TextContent, UriContent, Role

message = ChatMessage(
    role=Role.USER,
    contents=[
        TextContent(text="What do you see in this image?"),
        UriContent(
            uri="https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
            media_type="image/jpeg"
        )
    ]
)

result = await agent.run(message)
print(result.text)

The image depicts a serene landscape featuring a wooden pathway winding through lush green grass and a variety of foliage. The sky above is vibrant with soft clouds and a blue backdrop, suggesting a bright and pleasant day. The pathway seems to lead further into the natural scenery, inviting exploration of the surrounding vegetation. Overall, it presents a peaceful and scenic view of nature.


## Running the agent with a multi-turn conversation

Agents are stateless and do not maintain any state internally between calls. To have a multi-turn conversation with an agent, you need to create an object to hold the conversation state and pass this object to the agent when running it.

To create the conversation state object, call the GetNewThread method on the agent instance.

In [8]:
thread = agent.get_new_thread()

You can then pass this thread object to the run and run_stream methods on the agent instance, along with the user input.

In [11]:

result1 = await agent.run("Tell me a joke about a pirate.", thread=thread)
print(result1.text)

result2 = await agent.run("Now add some emojis to the joke and tell it in the voice of a pirate's parrot.", thread=thread)
print(result2.text)

Why did the pirate go to school?

Because he wanted to improve his "arrrticulation"!
Squawk! 🦜 Why did the pirate go to school? 📚

Because he wanted to improve his "arrrticulation"! 😂 Arrr!


List the messages stored i the `thread` object to see the full conversation history.

In [ ]:
for message in await thread.message_store.list_messages():
    print(f"{message.role}: {message.contents[0].text}")

user: Tell me a joke about a pirate.
assistant: Why did the pirate go to school?

Because he wanted to improve his "arrrticulation"!
user: Now add some emojis to the joke and tell it in the voice of a pirate's parrot.
assistant: Squawk! 🦜 Why did the pirate go to school? 📚

Because he wanted to improve his "arrrticulation"! 😂 Arrr!
